In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

MY_ID = "arpriya04"

# 1. Create the schema in your workspace catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.capstone_{MY_ID}")

# 2. Create the raw volume inside the schema
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.capstone_{MY_ID}.raw")

# 3. Define path variable
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

In [0]:
display(dbutils.fs.ls(VOL))

[]

In [0]:
import random
from datetime import datetime, timedelta
import pandas as pd

MY_ID = "arpriya04"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

# Generate mock data for Ride Cancellation Analysis
random.seed(42)
n_rows = 1000

cities = ["New York", "Los Angeles", "Chicago", "Houston", "Phoenix"]
cancellation_reasons = [
    "Driver too far",
    "Customer changed mind",
    "Long wait time",
    "Wrong address",
    None,
]
cancellation_by = ["Driver", "Rider", "System", None]

data = []
start_date = datetime(2026, 1, 1)

for i in range(1, n_rows + 1):
    ride_id = f"RIDE_{1000 + i}"
    # Introduce some duplicate keys for data cleaning practice
    if i % 100 == 0:
        ride_id = f"RIDE_{1000 + i - 1}"

    rider_id = f"USER_{random.randint(100, 500)}"
    driver_id = f"DRV_{random.randint(10, 50)}"
    city = random.choice(cities)

    ride_date = start_date + timedelta(
        days=random.randint(0, 180), minutes=random.randint(0, 1440)
    )
    ride_date_str = ride_date.strftime("%Y-%m-%d %H:%M:%S")

    # Introduce some invalid dates
    if i % 80 == 0:
        ride_date_str = "INVALID_DATE"

    is_cancelled = random.choice([True, False, False])

    if is_cancelled:
        cancelled_by = random.choice(["Driver", "Rider", "System"])
        cancel_reason = random.choice(
            ["Driver too far", "Customer changed mind", "Long wait time"]
        )
        fare = None
    else:
        cancelled_by = None
        cancel_reason = None
        fare = round(random.uniform(10.0, 150.0), 2)

    # Introduce corrupt fare string
    if i % 90 == 0:
        fare = "CORRUPT_FARE"

    data.append(
        {
            "ride_id": ride_id,
            "rider_id": rider_id,
            "driver_id": driver_id,
            "city": city,
            "ride_timestamp": ride_date_str,
            "is_cancelled": str(is_cancelled),
            "cancelled_by": cancelled_by,
            "cancellation_reason": cancel_reason,
            "fare_amount": str(fare) if fare is not None else "",
        }
    )

# Convert to Pandas and save as CSV directly in your Volume
pdf = pd.DataFrame(data)
pdf.to_csv(f"{VOL}/ride_cancellations.csv", index=False)

print(
    f"Data successfully generated and saved to {VOL}/ride_cancellations.csv"
)

Data successfully generated and saved to /Volumes/workspace/capstone_arpriya04/raw/ride_cancellations.csv


In [0]:
display(dbutils.fs.ls(VOL))

path,name,size,modificationTime
dbfs:/Volumes/workspace/capstone_arpriya04/raw/ride_cancellations.csv,ride_cancellations.csv,74773,1790790791000


In [0]:
# Read the generated CSV into PySpark
df_raw = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{VOL}/ride_cancellations.csv")
)

# Add metadata columns
df_bronze = (
    df_raw.withColumn("_source_file", F.lit("ride_cancellations.csv"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn(
        "_row_hash",
        F.sha2(
            F.concat_ws(
                "||", *[F.coalesce(F.col(c), F.lit("")) for c in df_raw.columns]
            ),
            256,
        ),
    )
)

# Write to Bronze Delta Table
df_bronze.write.format("delta").mode("overwrite").saveAsTable(
    f"workspace.capstone_{MY_ID}.bronze_ride_cancellations"
)

In [0]:
from pyspark.sql.window import Window

# 1. Read from Bronze Delta Table
df_bronze = spark.read.table(f"workspace.capstone_{MY_ID}.bronze_ride_cancellations")

# 2. Type casting & clean transformation
df_transformed = df_bronze.withColumn(
    "ride_timestamp_clean", F.expr("try_cast(ride_timestamp as timestamp)")
).withColumn(
    "is_cancelled_clean", F.expr("try_cast(is_cancelled as boolean)")
).withColumn(
    "fare_amount_clean", F.expr("try_cast(fare_amount as double)")
)

# 3. Deduplication using Window
window_spec = Window.partitionBy("ride_id").orderBy(F.col("_ingested_at").desc())
df_dedup = df_transformed.withColumn("row_num", F.row_number().over(window_spec))

# 4. Separate Valid Records vs Rejects
# Quality Rule: Valid records must have row_num == 1 AND valid ride_timestamp
valid_condition = (F.col("row_num") == 1) & (F.col("ride_timestamp_clean").isNotNull())

df_silver_valid = (
    df_dedup.filter(valid_condition)
    .select(
        "ride_id",
        "rider_id",
        "driver_id",
        "city",
        F.col("ride_timestamp_clean").alias("ride_timestamp"),
        F.col("is_cancelled_clean").alias("is_cancelled"),
        "cancelled_by",
        "cancellation_reason",
        F.col("fare_amount_clean").alias("fare_amount"),
        "_source_file",
        "_ingested_at",
        "_row_hash"
    )
)

# 5. Route Invalid/Duplicate rows to Rejects Table with reason
df_silver_rejects = (
    df_dedup.filter(~valid_condition)
    .withColumn(
        "rejection_reason",
        F.when(F.col("row_num") > 1, "Duplicate ride_id")
        .when(F.col("ride_timestamp_clean").isNull(), "Invalid or unparseable timestamp")
        .otherwise("Corrupt record")
    )
    .select(
        "ride_id",
        "ride_timestamp",
        "rejection_reason",
        "_source_file",
        "_ingested_at",
        "_row_hash"
    )
)

# 6. Write Silver Tables
df_silver_valid.write.format("delta").mode("overwrite").saveAsTable(
    f"workspace.capstone_{MY_ID}.silver_ride_cancellations"
)

df_silver_rejects.write.format("delta").mode("overwrite").saveAsTable(
    f"workspace.capstone_{MY_ID}.silver_rejects"
)

print("Silver layer successfully built!")

Silver layer successfully built!


In [0]:
# 1. Read from Silver Delta Table
df_silver = spark.read.table(f"workspace.capstone_{MY_ID}.silver_ride_cancellations")

# 2. Extract year-month and compute Gold business aggregations
df_gold = (
    df_silver.withColumn("ride_month", F.date_trunc("month", F.col("ride_timestamp")))
    .groupBy("city", "ride_month")
    .agg(
        F.count("ride_id").alias("total_rides"),
        F.sum(F.when(F.col("is_cancelled") == True, 1).otherwise(0)).alias("total_cancellations"),
        F.round(
            (F.sum(F.when(F.col("is_cancelled") == True, 1).otherwise(0)) / F.count("ride_id")) * 100,
            2
        ).alias("cancellation_rate_pct"),
        F.round(
            F.sum(F.when(F.col("is_cancelled") == False, F.col("fare_amount")).otherwise(0)),
            2
        ).alias("total_completed_revenue")
    )
    .orderBy("ride_month", "city")
)

# 3. Save as Gold Delta Table
df_gold.write.format("delta").mode("overwrite").saveAsTable(
    f"workspace.capstone_{MY_ID}.gold_ride_cancellation_metrics"
)

# 4. Export Gold table as a single CSV to your Volume export directory
export_path = f"{VOL}/export/gold_ride_cancellation_metrics.csv"

(
    df_gold.coalesce(1)
    .write.mode("overwrite")
    .option("header", "true")
    .csv(export_path)
)

print(f"Gold layer successfully created and exported to: {export_path}")

Gold layer successfully created and exported to: /Volumes/workspace/capstone_arpriya04/raw/export/gold_ride_cancellation_metrics.csv
